# Stage 6–8: Preprocessing, Train-Test Split and Model Building
We use a scikit-learn Pipeline so missing-value imputation, scaling and one-hot encoding are learned from training data only. This prevents preprocessing leakage.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

df = pd.read_csv(DATA_DIR / "used_phones_features.csv")
TARGET = "normalized_used_price"
y = pd.to_numeric(df[TARGET], errors="coerce")
X = df.drop(columns=[TARGET])
valid = y.notna() & (y > 0)
X, y = X.loc[valid].copy(), y.loc[valid].copy()
id_cols = [c for c in X.columns if c.lower() in {"id","phone_id","listing_id","url"}]
X = X.drop(columns=id_cols)
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
X = X.drop(columns=constant_cols)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=42
)
print(X_train.shape, X_test.shape)

(2763, 14) (691, 14)


In [2]:
numeric_cols = X_train.select_dtypes(include=["number","bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_cols),
    ("categorical", categorical_pipe, categorical_cols)
], sparse_threshold=0)

In [3]:
cv_results_df = pd.DataFrame(cv_results).sort_values("CV_RMSE")
display(cv_results_df)


NameError: name 'cv_results' is not defined

Choose the model with the lowest average cross-validation RMSE on the training partition. The test partition must remain untouched until final evaluation.